# M3 Agentic AI - 邮件助手工作流

## 1. 简介

### 1.1 实验概述

在本模块的早些时候，Andrew 展示了日历助手如何使用多种工具完成复杂任务的例子。在本实验中，你将探索一个类似的场景——**邮件助手智能体**。

该智能体工作流可以执行与邮件管理相关的多种任务，包括发送邮件、搜索特定发件人的邮件、删除邮件等。你只需用自然语言下达指令——例如「查看老板发来的未读邮件」或「删除 Happy Hour 那封邮件」——然后观察它如何选择合适的工具并完成任务。

<img src="lab_overview.png" alt="日历助手示例" width="700"/>

### 🎯 1.2 学习目标
完成本实验后，你将能够**把 LLM 与工具连接起来**，用自然语言下达指令，并观察智能体如何选择、执行和验证多步任务（如搜索、发送、删除邮件）。

## 2. 初始化环境与客户端

与之前的实验一样，你现在需要配置运行环境。

In [ ]:
# ================================
# 导入
# ================================

from dotenv import load_dotenv
import json

import utils
import display_functions
import email_tools
from config import get_settings
from llm_client import ZhipuToolClient

# ================================
# 环境与客户端
# ================================
load_dotenv()
settings = get_settings()
client = ZhipuToolClient(settings)
MODEL = settings.zhipu_model


## 3. 模拟邮件服务

### 3.1 组件
本实验使用**模拟邮件后端**来模仿真实世界的邮件交互。
你可以把它当作自己的沙盒收件箱：它预置了若干邮件，方便你练习而无需发送真实邮件。

你无需从零搭建该后端，而是使用工具直接与之交互。底层技术栈包括：

| 层级 | 作用 |
|-------------------------|--------------------------------|
| **FastAPI** | 提供 REST 接口 |
| **SQLite + SQLAlchemy** | 在本地存储和查询邮件 |
| **Pydantic** | 校验输入与输出 |
| **智谱工具调用** | 连接 LLM 与邮件服务 |


### 3.2 接口端点
该服务提供若干路由，模拟常见邮件操作。
后续这些接口会被封装为工具，供助手自动调用：

- `POST /send` → 发送新邮件
- `GET /emails` → 列出所有邮件
- `GET /emails/unread` → 仅显示未读邮件
- `GET /emails/{id}` → 按 ID 获取邮件
- `GET /emails/search?q=...` → 按关键词搜索
- `GET /emails/filter` → 按收件人或日期范围过滤
- `PATCH /emails/{id}/read` → 标记为已读
- `PATCH /emails/{id}/unread` → 标记为未读
- `DELETE /emails/{id}` → 按 ID 删除邮件
- `GET /reset_database` → 重置为初始数据（用于测试）


> 💡 **核心思路：** 这些端点相当于「收件箱控制台」。
> 接下来，你会把它们暴露为 Python 函数（工具），让 LLM 可以调用——把原始 API 路由变成智能体动作。

### 3.3 端点测试辅助函数
在把控制权交给 LLM 之前，你需要**先自己测试后端**。
`utils.test_*` 函数是对 API 端点的简单封装，让你无需手写 HTTP 请求即可尝试**发送、列表、搜索、过滤、标记、删除**等操作。

**提示：** 可通过顶部菜单 **File > Open** 打开 `utils.py` 文件。

每个辅助函数名称清晰易懂。在下一个代码单元格中，取消注释你想运行的那一行，查看输出，确认邮件服务行为符合预期。

例如，你可以测试：
- 发送测试邮件
- 按 ID 获取邮件
- 列出所有邮件
- 标记已读/未读
- 删除邮件

这一步是在把控制权交给智能体之前的**健全性检查**，确保模拟邮件服务正常运行，行为与真实邮件服务类似。

👉 要测试端点，请**取消注释**你想尝试的行，运行单元格并立即查看结果。

In [ ]:
# 取消注释你想尝试的 utils.test_* 行
new_email_id = utils.test_send_email()
_ = utils.test_get_email(new_email_id['id'])
#_ = utils.test_list_emails()
#_ = utils.test_filter_emails(recipient="test@example.com")
#_ = utils.test_search_emails("lunch")
#_ = utils.test_unread_emails()
#_ = utils.test_mark_read(new_email_id['id'])
#_ = utils.test_mark_unread(new_email_id['id'])
#_ = utils.test_delete_email(new_email_id['id'])
#_ = utils.reset_database()


## 4. 邮件智能体的工具层

### 4.1 为什么需要工具？
端点验证通过后，下一步是把它们以 Python 函数的形式暴露给 LLM，这些函数称为**工具（tools）**。每个工具封装一条 REST 路由，把原始 API 调用变成智能体可执行的动作——如列表、阅读、搜索、发送、删除或切换已读状态。

> 可以把工具理解为智能体的**执行器**：你给出自然语言指令（「查看老板发来的未读邮件并礼貌回复」），模型会选择**调用哪些工具**以及**以什么顺序**完成任务。

### 4.2 设计建议
- 工具 **docstring** 应简短、祈使、且明确描述动作。
- 返回**一致、紧凑的 JSON**，便于模型串联多步结果。
- 每个工具**职责单一**（一条路由、一种效果）。

### 4.3 可用工具
| 工具函数 | 动作 |
|------------------------------------|------------------------------------------------------------------------|
| `list_all_emails()` | 获取所有邮件，按时间倒序 |
| `list_unread_emails()` | 仅获取未读邮件 |
| `search_emails(query)` | 在主题、正文或发件人中搜索关键词 |
| `filter_emails(...)` | 按收件人和/或日期范围过滤 |
| `get_email(email_id)` | 按 ID 获取单封邮件 |
| `mark_email_as_read(id)` | 标记为已读 |
| `mark_email_as_unread(id)` | 标记为未读 |
| `send_email(...)` | 发送一封（模拟）邮件 |
| `delete_email(id)` | 按 ID 删除邮件 |
| `search_unread_from_sender(addr)` | 返回某发件人的未读邮件（如 `boss@email.com`） |

**提示：** 可通过顶部菜单 **File > Open** 打开 `email_tools.py` 文件。

例如，**测试 `list_all_emails()` 工具：**
```python
    all_emails = email_tools.list_all_emails()
```

现在，让我们尝试几个连接模拟端点的工具，确保一切正常工作。

👉 **取消注释**你想运行的行，执行单元格并查看输出。

In [ ]:
# 测试发送邮件并按 ID 获取
new_email = email_tools.send_email("test@example.com", "午餐计划", "中午见面好吗？")
content_ = email_tools.get_email(new_email['id'])

# 取消注释你想尝试的项：
#content_ = email_tools.list_all_emails()
#content_ = email_tools.list_unread_emails()
#content_ = email_tools.search_emails("lunch")
#content_ = email_tools.filter_emails(recipient="test@example.com")
#content_ = email_tools.mark_email_as_read(new_email['id'])
#content_ = email_tools.mark_email_as_unread(new_email['id'])
#content_ = email_tools.search_unread_from_sender("test@example.com")
#content_ = email_tools.delete_email(new_email['id'])

utils.print_html(content=json.dumps(content_, indent=2), title="测试 email_tools")


## 5. 准备智能体 Prompt

在给邮件助手分配任务之前，你需要创建一个名为 `build_prompt()` 的小辅助函数。
该函数用系统风格的引导语包装自然语言请求，使 LLM：

- 认识到自己正在扮演**邮件助手智能体**
- 理解可以使用已注册的工具
- 直接执行操作，无需征求确认（无人工介入）

In [ ]:
def build_prompt(request_: str) -> str:
    return f"""
- 你是一个专门管理邮件的 AI 助手。
- 你可以执行列表、搜索、过滤和操作邮件等多种动作。
- 使用提供的工具与邮件系统交互。
- 执行操作前不要向用户征求确认。
- 如需使用，我的邮箱地址是 "you@email.com"，可用于发送邮件或执行与我账户相关的操作。

{request_.strip()}
"""


运行下一个单元格，查看上述函数如何**把你的原始用户指令**包装上系统说明。
例如：

In [ ]:
example_prompt = build_prompt("删除 Happy Hour 那封邮件")
utils.print_html(content=example_prompt, title="示例 Prompt")


### 5.3 重置邮件服务

由于你一直在实验邮件服务，现在把它重置回初始状态。

调用以下工具函数即可清空并刷新模拟邮件服务：

In [6]:
utils.reset_database()

{'message': 'Database reset and emails reloaded'}

## 6. LLM + 邮件工具

### 6.1 场景
到目前为止，你一直在直接与后端交互。现在是时候让 **LLM 接管**，作为你的邮件助手智能体。

例如，你可以这样请求：
> 「查看 `boss@email.com` 发来的未读邮件，标记为已读，并发送一封礼貌的跟进回复。」

### 6.2 会发生什么
1. 智能体理解你的指令。
2. 它选择合适的工具（`search_unread_from_sender` → `mark_email_as_read` → `send_email`）。
3. 它自动执行每一步，无需征求确认。

`ZhipuToolClient` 负责工具 schema 暴露、参数绑定、执行与结果传递——让你专注于智能体**完成了什么**，而不是**如何调用 API**。

### 6.3 运行
运行下一个单元格，观察智能体如何编排多个工具完成你的请求。你也可以尝试自己的邮件相关指令。

*关注以下几点：*
- 清晰的**工具调用轨迹**，显示使用了哪些工具及传入的参数。
- 简洁的**最终回复**，总结已执行的操作（如「找到 1 封未读邮件，已标记为已读，已发送跟进邮件」）。

In [ ]:
# 尝试你自己的请求
prompt_ = build_prompt("查看 boss@email.com 发来的未读邮件，标记为已读，并发送一封礼貌的跟进回复。")

response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": prompt_}],
    tools=[
        email_tools.search_unread_from_sender,
        email_tools.list_unread_emails,
        email_tools.search_emails,
        email_tools.get_email,
        email_tools.mark_email_as_read,
        email_tools.send_email
    ],
    max_turns=5,
)

display_functions.pretty_print_chat_completion(response)


### 6.4 缺少工具：`delete_email`

如果所需的工具不可用，会发生什么？

例如，尝试这个请求：
> 「删除 alice@work.com 的邮件。」

由于未注册 `delete_email` 工具，LLM 仍会尝试回复，但无法完成删除操作。

<div style="background-color: #ffe4e1; padding: 12px; border-radius: 6px; color: black;">
  <h4>🔍 关键启示</h4>
  <p style="margin: 0;">
    这强调了一点：<b>你提供的工具集合直接决定了智能体能做什么。</b>
  </p>
</div>


**当前可用工具**

| 工具函数 | 动作 |
|------------------------------------|------------------------------------------------------------------------|
| `list_all_emails()` | 获取所有邮件，按时间倒序 |
| `list_unread_emails()` | 仅获取未读邮件 |
| `search_emails(query)` | 在主题、正文或发件人中搜索关键词 |
| `filter_emails(...)` | 按收件人和/或日期范围过滤 |
| `get_email(email_id)` | 按 ID 获取单封邮件 |
| `mark_email_as_read(id)` | 标记为已读 |
| `mark_email_as_unread(id)` | 标记为未读 |
| `send_email(...)` | 发送一封（模拟）邮件 |
| `delete_email(id)` | 按 ID 删除邮件 |
| `search_unread_from_sender(addr)` | 返回某发件人的未读邮件（如 `boss@email.com`） |

In [ ]:
# 尝试调用不可用工具的请求
prompt_ = build_prompt("删除 alice@work.com 的邮件")

response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": prompt_}],
    tools=[
        email_tools.search_unread_from_sender,
        email_tools.list_unread_emails,
        email_tools.search_emails,
        email_tools.get_email,
        email_tools.mark_email_as_read,
        email_tools.send_email
    ],
    max_turns=5
)

display_functions.pretty_print_chat_completion(response)


#### 6.4.1 启用 `delete_email` 后重试

如果所需的工具不可用，会发生什么？

在上一步中，你给 LLM 提供了以下工具列表：
```python
    tools=[
        email_tools.search_unread_from_sender,
        email_tools.list_unread_emails,
        email_tools.search_emails,
        email_tools.get_email,
        email_tools.mark_email_as_read,
        email_tools.send_email
    ]
```

由于该列表中没有 `delete_email`，智能体无法删除邮件。

现在把 `delete_email` 加入工具列表并重新运行单元格。这次智能体具备完成任务所需的全部能力。

> 提示：观察调用顺序——找到目标邮件后，智能体应调用 `delete_email` 完成删除。

In [ ]:
prompt_ = build_prompt("删除 alice@work.com 的邮件")

response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": prompt_}],
    tools=[
        email_tools.search_unread_from_sender,
        email_tools.list_unread_emails,
        email_tools.search_emails,
        email_tools.get_email,
        email_tools.mark_email_as_read,
        email_tools.send_email,
        email_tools.delete_email
    ],
    max_turns=5
)

display_functions.pretty_print_chat_completion(response)


### 6.5 定向操作：删除「Happy Hour」邮件

测试收件箱预置了一封标题为 **「Happy Hour」** 的邮件。你的任务是 指示智能体找到并删除它。

参考数据如下：

```json
{
  "id": 1,
  "sender": "eric@work.com",
  "recipient": "you@email.com",
  "subject": "Happy Hour",
  "body": "We're planning drinks this Friday!",
  "timestamp": "2025-06-13T04:48:59.096908",
  "read": false
}
```

运行下一个单元格，观察智能体如何搜索并删除该邮件。

In [ ]:
prompt_ = build_prompt("删除 happy hour 那封邮件")

response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": prompt_}],
    tools=[
        email_tools.search_unread_from_sender,
        email_tools.list_unread_emails,
        email_tools.search_emails,
        email_tools.get_email,
        email_tools.mark_email_as_read,
        email_tools.send_email,
        email_tools.delete_email
    ],
    max_turns=5
)

display_functions.pretty_print_chat_completion(response)


## 7. 总结

- 在本实验中，**你探索了** LLM 邮件智能体如何通过工具调用与模拟邮件服务交互。
- **工具调用**让 LLM 超越文本生成——能够调用函数（工具）并完成多步任务。
- 可用工具集合决定了智能体能做什么、不能做什么（例如没有 `delete_email` 就无法删除邮件）。
- 清晰的 docstring 和一致的行为有助于 LLM 在每一步选择正确的工具。
- `ZhipuToolClient` 管理交互层：把 Python 函数暴露为工具、接收参数、发起 API 请求并返回结果。
- 观察完整工作流——从 prompt → 工具调用 → 输出 → 最终回复——是理解和改进智能体推理与行动方式的关键。

<div style="border:1px solid #22c55e; border-left:6px solid #16a34a; background:#dcfce7; border-radius:6px; padding:14px 16px; color:#064e3b; font-family:system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,Cantarell,Noto Sans,sans-serif;">

🎉 <strong>恭喜！</strong>

你刚刚引导一个由 LLM 驱动的<em>邮件助手智能体</em>，把自然语言请求转化为对模拟邮件服务的具体操作。你看到了工具如何把普通语言变成可靠操作——列表、搜索、标记已读、发送、删除——而无需你亲自处理原始 HTTP 请求。

你也了解到，暴露的工具决定了智能体的真实能力。缺少工具时，智能体可以推理但无法行动；工具齐全时，它可以串联步骤、传递参数，并准确完成你的请求。在此过程中，`ZhipuToolClient` 处理了底层细节：schema 暴露、参数绑定、执行与结果传递——让你专注于「要做什么」，而不是「如何接线」。

掌握这套工作流后，你已能设计面向任务的智能体：安全透明地行动、说明自己做了什么，并从简单 prompt 扩展到你自己服务上的可靠多步自动化。 🌟
</div>